In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# DVD Rental Store Analysis — SQL & SQLite
## York University — Big Data Analytics Certificate (2024)

**Author:** Lalita  
**Tools:** Python, SQLite, Pandas, Matplotlib, Seaborn  
**Database:** SQLite Sakila Sample Database (16 tables)  

---

## Project Overview
This project analyzes a DVD rental store database using 
complex SQL JOINs across 6 tables to identify top performing 
films, most valuable customers and highest revenue categories.

## Key Findings
| Insight | Result |
|---|---|
| Top Rented Film | BUCKET BROTHERHOOD — 34 rentals |
| Top Customer | ELEANOR HUNT — 46 rentals |
| Highest Revenue Category | Sports — $5,314.21 |
| SQL Tables Used | 6 tables with complex JOINs |

## SQL Queries Used
- Complex JOINs across film, inventory, rental, payment tables
- GROUP BY and ORDER BY for aggregation
- COUNT and SUM for business metrics

## Conclusion
SQL analysis revealed Sports as highest revenue category
and identified top customers and films to support
data driven business decisions for the rental store.

---
🔗 **Kaggle Profile:** kaggle.com/lalitacanada  
🎓 **York University — Big Data Analytics (2024)**

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("atanaskanev/sqlite-sakila-sample-database")

print("Path to dataset files:", path)

In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Connect to SQLite database
conn = sqlite3.connect('/kaggle/input/datasets/atanaskanev/sqlite-sakila-sample-database/sqlite-sakila.db')

# Show all tables
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table'", conn)
print("Tables in database:")
print(tables)

In [ ]:
# Query 1 — Top 10 most rented films
query1 = """
SELECT f.title, 
       COUNT(r.rental_id) as total_rentals
FROM film f
JOIN inventory i ON f.film_id = i.film_id
JOIN rental r ON i.inventory_id = r.inventory_id
GROUP BY f.title
ORDER BY total_rentals DESC
LIMIT 10
"""
top_films = pd.read_sql_query(query1, conn)
print("Top 10 Most Rented Films:")
print(top_films)

In [ ]:
# Query 2 — Top 10 customers by rentals
query2 = """
SELECT c.first_name || ' ' || c.last_name as customer_name,
       COUNT(r.rental_id) as total_rentals,
       SUM(p.amount) as total_spent
FROM customer c
JOIN rental r ON c.customer_id = r.customer_id
JOIN payment p ON r.rental_id = p.rental_id
GROUP BY c.customer_id
ORDER BY total_rentals DESC
LIMIT 10
"""
top_customers = pd.read_sql_query(query2, conn)
print("Top 10 Customers by Rentals:")
print(top_customers)

In [ ]:
# Query 3 — Revenue by film category
query3 = """
SELECT c.name as category,
       COUNT(r.rental_id) as total_rentals,
       SUM(p.amount) as total_revenue
FROM category c
JOIN film_category fc ON c.category_id = fc.category_id
JOIN film f ON fc.film_id = f.film_id
JOIN inventory i ON f.film_id = i.film_id
JOIN rental r ON i.inventory_id = r.inventory_id
JOIN payment p ON r.rental_id = p.rental_id
GROUP BY c.name
ORDER BY total_revenue DESC
"""
category_revenue = pd.read_sql_query(query3, conn)
print("Revenue by Film Category:")
print(category_revenue)

In [ ]:
# Visualization 1 — Top 10 Most Rented Films
plt.figure(figsize=(12,6))
sns.barplot(data=top_films, 
            x='total_rentals', 
            y='title',
            hue='title',
            legend=False,
            palette='Blues_r')
plt.title('Top 10 Most Rented Films')
plt.xlabel('Total Rentals')
plt.ylabel('Film Title')
plt.show()

# Visualization 2 — Revenue by Category
plt.figure(figsize=(12,6))
sns.barplot(data=category_revenue,
            x='total_revenue',
            y='category',
            hue='category',
            legend=False,
            palette='viridis')
plt.title('Total Revenue by Film Category')
plt.xlabel('Total Revenue ($)')
plt.ylabel('Category')
plt.show()

In [ ]:
print("=" * 50)
print("KEY INSIGHTS — DVD Rental Store Analysis")
print("=" * 50)

print("\n📊 Database Overview:")
print("Database: SQLite Sakila Sample Database")
print("Tables analyzed: film, customer, rental,")
print("                 payment, category, inventory")

print("\n🎬 Top Rented Film:")
print(f"  {top_films.iloc[0]['title']} — {top_films.iloc[0]['total_rentals']} rentals")

print("\n👤 Top Customer:")
print(f"  {top_customers.iloc[0]['customer_name']}")
print(f"  Rentals: {top_customers.iloc[0]['total_rentals']}")
print(f"  Total Spent: ${top_customers.iloc[0]['total_spent']:.2f}")

print("\n💰 Top Revenue Category:")
print(f"  {category_revenue.iloc[0]['category']}")
print(f"  Revenue: ${category_revenue.iloc[0]['total_revenue']:.2f}")
print(f"  Rentals: {category_revenue.iloc[0]['total_rentals']}")

print("\n💡 Key Findings:")
print("1. Sports generates highest revenue — $5,314")
print("2. BUCKET BROTHERHOOD is most rented film")
print("3. ELEANOR HUNT is top customer — 46 rentals")
print("4. Complex SQL JOINs across 6 tables used")

print("\n✅ Conclusion:")
print("SQL analysis of DVD rental database revealed")
print("key business insights on top performing films,")
print("most valuable customers and highest revenue")
print("categories to support business decisions.")

conn.close()